# Modelling

These models will be tested:

| Model       | Features                                                | Reason                                     |
| ----------- | ------------------------------------------------------- | -------------------------------------------|
| RFM         | Recency, Frequency, Monetary                            | Classical Behaviour Baseline               |
| RFM_NLP     | RFM + TF-IDF-Product Text Features (NLP)                | Test on Product Preferences                | 
| RFM_NLP_BEH | RFM + NLP + information on time and preferences         | Test on broader Customer-Behaviour-Profile |

In [3]:
# Imports

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [4]:
# Load Data

df = pd.read_csv('../01_data/02_processed_data/customer_profile.csv')
df.head()


,CustomerID,recency,frequency,monetary,weekend_share,avg_purchase_hour,preferred_hour,active_purchase_days,preferred_time_5cat,avg_days_between_orders,...,text_svd_05,text_svd_06,text_svd_07,text_svd_08,text_svd_09,text_svd_10,text_svd_11,text_svd_12,text_svd_13,text_svd_14
0,12346.0,326,12,77556.46,0.000,10.833333,13,8,midday,36.369129,...,-0.054787,-0.021104,0.195492,0.084543,0.104115,-0.240231,0.245742,0.311689,0.199950,-0.054729
1,12347.0,2,8,5633.32,0.125,12.500000,14,8,afternoon,57.437698,...,-0.112825,0.064833,-0.152168,0.155788,-0.040855,-0.156677,0.061693,-0.027149,-0.060058,0.173658
2,12348.0,75,5,2019.40,0.200,13.200000,10,5,midday,90.731597,...,-0.161823,0.537691,-0.191536,0.081759,-0.091979,0.089767,-0.054764,0.083323,0.009273,0.111751
3,12349.0,19,4,4428.69,0.000,9.750000,9,4,morning,190.284954,...,-0.186275,-0.166294,0.324660,-0.083427,0.013956,-0.200510,0.100400,0.139637,-0.129949,-0.048644
4,12350.0,310,1,334.40,0.000,16.000000,16,1,afternoon,NaN,...,0.124714,-0.115467,0.113825,-0.004743,-0.085082,0.096691,0.024741,-0.027823,-0.030778,-0.002321


### Baseline RFM Modell

In [5]:
# Features Selection - Baseline RFM

rfm=['recency', 'frequency', 'monetary']

rfm_log=[]

for col in rfm: 
    new_col = f"{col}_log"
    df[new_col] = np.log1p(df[col])
    rfm_log.append(new_col)

rfm_log


['recency_log', 'frequency_log', 'monetary_log']

In [6]:
# Clustering

X = df[rfm_log]

preprocess = ColumnTransformer([
    ('scaler', StandardScaler(), rfm_log)
])

X_processed = preprocess.fit_transform(X)

n_components_to_test = range(2, 10)

inertias = []
silhouettes = []

for n in n_components_to_test:
    model = KMeans(n_clusters=n, random_state=42, n_init="auto")
    labels = model.fit_predict(X_processed)

    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_processed, labels))

results = pd.DataFrame({
    'k': list(n_components_to_test),
    'inertia': inertias,
    'silhouette': silhouettes
})


In [7]:
results

,k,inertia,silhouette
0,2,8620.579019,0.438884
1,3,6386.969759,0.347550
2,4,4955.740123,0.364701
3,5,4129.548196,0.343958
4,6,3584.091879,0.332019
5,7,3221.601853,0.316131
6,8,2932.791710,0.305595
7,9,2692.710339,0.294146


### RFM Model with NLP
#### KMeans

In [8]:
svd = [f'text_svd_{i:02d}' for i in range(15)]

X = df[rfm_log + svd] 

preprocess = ColumnTransformer([
    ('scaler', StandardScaler(), rfm_log),
    ('passthrough', 'passthrough', svd) 
])

X_processed = preprocess.fit_transform(X)

n_components_to_test = range(2, 10)

inertias = []
silhouettes = []

for n in n_components_to_test:
    model = KMeans(n_clusters=n, random_state=42, n_init="auto")
    labels = model.fit_predict(X_processed)

    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_processed, labels))

results = pd.DataFrame({
    'k': list(n_components_to_test),
    'inertia': inertias,
    'silhouette': silhouettes
})

results

,k,inertia,silhouette
0,2,10774.457651,0.380496
1,3,8524.470773,0.275843
2,4,7078.974717,0.280112
3,5,6247.588425,0.248463
4,6,5698.384774,0.231226
5,7,5334.277116,0.192812
6,8,5040.589912,0.200481
7,9,4794.322214,0.185466


#### DBSCAN

In [ ]:
# --- Preparation ---
rfm_log = ['recency_log', 'frequency_log', 'monetary_log']
svd = [f'text_svd_{i:02d}' for i in range(15)]

# RFM-only
X_rfm = df[rfm_log]
X_rfm_scaled = StandardScaler().fit_transform(X_rfm)

# RFM+NLP
X_both = df[rfm_log + svd]
X_both_scaled = StandardScaler().fit_transform(X_both)

# --- DBSCAN for RFM-only ---
dbscan_rfm = DBSCAN(
    eps=0.5,           # Nachbarschaftsradius
    min_samples=5,     # Mindestpunkte pro Cluster
    n_jobs=-1,         # Alle CPU-Kerne
    metric='euclidean'
)
labels_rfm = dbscan_rfm.fit_predict(X_rfm_scaled)

# Results
n_clusters_rfm = len(set(labels_rfm)) - (1 if -1 in labels_rfm else 0)
n_noise_rfm = list(labels_rfm).count(-1)

print(f"RFM-only: {n_clusters_rfm} Cluster, {n_noise_rfm} Noise-Punkte")

# Silhouette (nur für Nicht-Noise-Punkte)
if n_clusters_rfm > 1:
    mask_rfm = labels_rfm != -1
    sil_rfm = silhouette_score(X_rfm_scaled[mask_rfm], labels_rfm[mask_rfm])
    print(f"  Silhouette: {sil_rfm:.3f}")

# --- DBSCAN for RFM+NLP ---
dbscan_both = DBSCAN(
    eps=0.5,
    min_samples=5,
    n_jobs=-1,
    metric='euclidean'
)
labels_both = dbscan_both.fit_predict(X_both_scaled)

n_clusters_both = len(set(labels_both)) - (1 if -1 in labels_both else 0)
n_noise_both = list(labels_both).count(-1)

print(f"RFM+NLP: {n_clusters_both} Cluster, {n_noise_both} Noise-Punkte")

if n_clusters_both > 1:
    mask_both = labels_both != -1
    sil_both = silhouette_score(X_both_scaled[mask_both], labels_both[mask_both])
    print(f"  Silhouette: {sil_both:.3f}")

RFM-only: 1 Cluster, 64 Noise-Punkte
RFM+NLP: 1 Cluster, 5862 Noise-Punkte
